# Equipment Simulation - Bonny Island, Nigeria

This notebook simulates telemetry data for industrial equipment at Bonny Island, Nigeria:
- **10 Gas Turbines**
- **10 Centrifugal Compressors**
- **30 Centrifugal Pumps**

**Simulation Parameters:**
- Start Date: July 1, 2025
- Duration: 6 months
- Sample Interval: 5 minutes
- Location: Bonny Island, Nigeria
- Increased failure rates for testing

**Output:**
- Telemetry records
- Failure records
- Database insertion

In [ ]:
import sys
import os
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import logging
from data_simulation.gas_turbine import GasTurbine
from data_simulation.centrifugal_compressor import CentrifugalCompressor
from data_simulation.centrifugal_pump import CentrifugalPump
from data_simulation.physics.environmental_conditions import EnvironmentalConditions, LocationType
from data_simulation.physics.weather_api_client import create_hybrid_environment
from data_simulation.ml_utils.ml_output_modes import OutputMode
from ingestion.equipment_sim import simulate_equipment
from ingestion.data_pipeline import DataPipeline
import sqlite3
# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# Simulation parameters
CONFIG = {
    # Equipment counts
    'n_turbines': 10,
    'n_compressors': 10,
    'n_pumps': 30,
    
    # Time parameters
    'start_date': datetime(2025, 7, 1, 0, 0, 0),  # July 1, 2025
    'duration_days': 183,  
    'sample_interval_min': 5,  
    
    # Location (Bonny Island, Nigeria)
    'location': 'Bonny Island',

    # Health parameters
    'health_range': (0.80, 0.99),  
    'degradation_multiplier': 1.0,

    # Random seed for reproducibility
    'random_seed': 42
}

# Set random seed
random.seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

print("Configuration:")
print(f"  Equipment: {CONFIG['n_turbines']} turbines, {CONFIG['n_compressors']} compressors, {CONFIG['n_pumps']} pumps")
print(f"  Duration: {CONFIG['duration_days']} days starting {CONFIG['start_date'].strftime('%Y-%m-%d')}")
print(f"  Sample interval: {CONFIG['sample_interval_min']} minutes")
print(f"  Location: {CONFIG['location']}")
print(f"  Samples per equipment: {CONFIG['duration_days'] * 24 * 60 // CONFIG['sample_interval_min']:,}")
print(f"  Total samples: {(CONFIG['n_turbines'] + CONFIG['n_compressors'] + CONFIG['n_pumps']) * CONFIG['duration_days'] * 24 * 60 // CONFIG['sample_interval_min']:,}")

In [ ]:
# Create environmental model for Bonny Island using REAL weather data
print("\nSetting up Weather API for Bonny Island...")
print("Note: This uses real weather data. Set use_real_weather=False for synthetic data.")

# Create hybrid environment with real weather and synthetic fallback
fallback = EnvironmentalConditions(LocationType.TROPICAL)

bonny_island_env = create_hybrid_environment(
use_real_weather=True,
    api_provider="weatherapi",
    api_key = os.getenv('WEATHER_API_KEY'),
    location_name="Port Harcourt",  
    country="Nigeria",
    fallback_source=fallback,  
    cache_enabled=True
)
# This fetches hourly weather once, then uses cached data for 5-min interpolation
print(f"Pre-caching weather data for {CONFIG['duration_days']} days...")
bonny_island_env.preload_cache(
    start_date=CONFIG['start_date'],
    end_date=CONFIG['start_date'] + timedelta(days=CONFIG['duration_days']),
    interval_hours=1  
)

# Test environmental conditions
test_conditions = bonny_island_env.get_conditions(
    elapsed_hours=0, 
    timestamp=CONFIG['start_date']
)
print("\nBonny Island Environmental Conditions (July 1, 2025):")
print(f"  Temperature: {test_conditions['ambient_temp_C']:.1f}°C (REAL DATA)")
print(f"  Humidity: {test_conditions['humidity_percent']:.1f}%")
print(f"  Pressure: {test_conditions['pressure_kPa']:.1f} kPa")
# print(f"  Wind Speed: {test_conditions['wind_speed_m_s']:.1f} m/s")
print(f"  Corrosion factor: {test_conditions['corrosion_factor']:.2f}x (coastal)")
print(f"  Fouling factor: {test_conditions['fouling_factor']:.2f}x")

In [ ]:
conn = sqlite3.connect("weather_cache.db")

env_model = pd.read_sql_query(
    "SELECT * FROM weather_cache",
    conn
)

conn.close()

# Convert timestamps
env_model["timestamp"] = pd.to_datetime(env_model["timestamp"], utc=True)

# Select July 1, 2025 at 00:00
row = env_model.loc[env_model["timestamp"] == "2025-12-25T14:00:00Z"].iloc[0]

# print("\nBonny Island Environmental Conditions (July 1, 2025):")
print(f"  Temperature: {row['ambient_temp_C']:.1f}°C (REAL DATA)")
print(f"  Humidity: {row['humidity_percent']:.1f}%")
print(f"  Pressure: {row['pressure_kPa']:.1f} kPa")

In [ ]:
EQUIPMENT_COMPONENTS = {
    "turbine": ["hgp", "blade", "bearing", "fuel"],
    "compressor": ["impeller", "bearing"],
    "pump": ["impeller", "seal", "bearing_de", "bearing_nde"]
}

# Equipment-specific health ranges
EQUIPMENT_HEALTH_RANGES = {
    "turbine": (0.80, 0.99),
    "compressor": (0.92, 0.99),  
    "pump": (0.80, 0.99)
}


def generate_initial_health(config, equipment_type):
    """
    Generate random initial health values for equipment components.
    Uses equipment-specific health ranges to account for different degradation models.
    """
    if equipment_type not in EQUIPMENT_COMPONENTS:
        raise ValueError(f"Unknown equipment type: {equipment_type}")

    # Use equipment-specific range if available, otherwise fall back to config
    low, high = EQUIPMENT_HEALTH_RANGES.get(equipment_type, config["health_range"])
    components = EQUIPMENT_COMPONENTS[equipment_type]

    return {
        comp: random.uniform(low, high)
        for comp in components
    }

In [ ]:
# Seed master data for all equipment using DataPipeline
print("SEEDING MASTER DATA FOR ALL EQUIPMENT")

try:
    # Initialize data pipeline
    db_url = os.getenv('POSTGRES_URL')
    pipeline = DataPipeline(db_url)
    pipeline.connect()
    
    print("\nDatabase connection successful\n")
    
    # Seed master data
    turbine_ids, compressor_ids, pump_ids = pipeline.seed_master_data(
        turbine_count=CONFIG['n_turbines'],
        compressor_count=CONFIG['n_compressors'],
        pump_count=CONFIG['n_pumps']
    )
    

    print("MASTER DATA SEEDING COMPLETE")
    print(f"\nTotal equipment created: {len(turbine_ids) + len(compressor_ids) + len(pump_ids)}")
    print(f"  - Gas Turbines: {len(turbine_ids)} (IDs: {turbine_ids[:5]}{'...' if len(turbine_ids) > 5 else ''})")
    print(f"  - Compressors: {len(compressor_ids)} (IDs: {compressor_ids[:5]}{'...' if len(compressor_ids) > 5 else ''})")
    print(f"  - Pumps: {len(pump_ids)} (IDs: {pump_ids[:6]}{'...' if len(pump_ids) > 6 else ''})")
    
    # Retrieve and display sample master data
    print("SAMPLE MASTER DATA FROM DATABASE")
    
    # Get turbine configs
    turbine_configs = pipeline.master_data.get_configs(turbine_ids[:3], 'turbine')
    print("\nGas Turbines (first 3):")
    df_turbines = pd.DataFrame(turbine_configs)
    display(df_turbines)
    
    # Get compressor configs
    compressor_configs = pipeline.master_data.get_configs(compressor_ids[:3], 'compressor')
    print("\nCentrifugal Compressors (first 3):")
    df_compressors = pd.DataFrame(compressor_configs)
    display(df_compressors)
    
    # Get pump configs
    pump_configs = pipeline.master_data.get_configs(pump_ids[:3], 'pump')
    print("\nCentrifugal Pumps (first 3):")
    df_pumps = pd.DataFrame(pump_configs)
    display(df_pumps)
    
    # Close database connection
    pipeline.close()
    print("\nDatabase connection closed")
    
except Exception as e:
    print(f"\nError: {e}")
    print("Please ensure PostgreSQL is running and credentials are correct")
    raise

In [ ]:
print(f"\nSimulating {CONFIG['n_turbines']} Gas Turbines...")
turbine_telemetry = []
turbine_failures = []

for i in range(CONFIG['n_turbines']):
    turbine_id = i + 1
    
    # Create turbine with reduced initial health
    initial_health = generate_initial_health(CONFIG, 'turbine')
    print(initial_health)
    turbine = GasTurbine(
        name=f"GT-{turbine_id:03d}",
        initial_health=initial_health,
        env_model=env_model,
        enable_environmental=True,
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True,
        output_mode = OutputMode.DERIVED_FEATURES
    )
    
    # logger.info(f"Simulating GT-{turbine_id:03d} (health: {initial_health})")
    
    # Simulate using extended function with degradation multiplier
    for record in simulate_equipment(
        turbine, 
        turbine_id, 
        'turbine',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True
    ):
        if record['type'] == 'failure':
            turbine_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
            # logger.info(f"GT-{turbine_id:03d} failed at {record['failure_time']} with code: {record['failure_mode_code']}")
            break
        else:
            turbine_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                **record['state']
            })
    
    print(f"  GT-{turbine_id:03d}: {len([r for r in turbine_telemetry if r['equipment_id'] == turbine_id]):,} samples, {'FAILED' if any(f['equipment_id'] == turbine_id for f in turbine_failures) else 'RUNNING'}")

print(f"\nTotal turbine telemetry: {len(turbine_telemetry):,} records")
print(f"Total turbine failures: {len(turbine_failures)}")

In [ ]:
print(f"\nSimulating {CONFIG['n_compressors']} Centrifugal Compressors...")
compressor_telemetry = []
compressor_failures = []

for i in range(CONFIG['n_compressors']):
    compressor_id = i + 1
    
    # Create compressor with reduced initial health
    initial_health = generate_initial_health(CONFIG, 'compressor')
    
    compressor = CentrifugalCompressor(
        name=f"CC-{compressor_id:03d}",
        initial_health=initial_health,
        design_flow=random.uniform(1200, 1800),
        design_head=random.uniform(7000, 9000),
        env_model=env_model,
        enable_environmental=True,
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True,
        output_mode = OutputMode.DERIVED_FEATURES
    )
    
    # logger.info(f"Simulating CC-{compressor_id:03d} (health: {initial_health})")
    
    # Simulate using extended function with degradation multiplier
    for record in simulate_equipment(
        compressor, 
        compressor_id, 
        'compressor',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True
    ):
        if record['type'] == 'failure':
            compressor_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
            # logger.info(f"CC-{compressor_id:03d} failed at {record['failure_time']} with code: {record['failure_mode_code']}")
            break
        else:
            compressor_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                **record['state']
            })
    
    print(f"  CC-{compressor_id:03d}: {len([r for r in compressor_telemetry if r['equipment_id'] == compressor_id]):,} samples, {'FAILED' if any(f['equipment_id'] == compressor_id for f in compressor_failures) else 'RUNNING'}")

print(f"\nTotal compressor telemetry: {len(compressor_telemetry):,} records")
print(f"Total compressor failures: {len(compressor_failures)}")

In [ ]:
print(f"\nSimulating {CONFIG['n_pumps']} Centrifugal Pumps...")
pump_telemetry = []
pump_failures = []

pump_services = [
    {'name': 'Crude Booster', 'design_flow': 200, 'design_head': 100, 'density': 850},
    {'name': 'Seawater Injection', 'design_flow': 300, 'design_head': 150, 'density': 1025},
    {'name': 'Process Water', 'design_flow': 100, 'design_head': 60, 'density': 1000},
    {'name': 'Methanol Pump', 'design_flow': 50, 'design_head': 80, 'density': 790},
    {'name': 'Fire Water', 'design_flow': 400, 'design_head': 120, 'density': 1000},
]

for i in range(CONFIG['n_pumps']):
    pump_id = i + 1
    service = pump_services[i % len(pump_services)]
    
    # Create pump with reduced initial health
    initial_health = generate_initial_health(CONFIG, 'pump')
    
    pump = CentrifugalPump(
        name=f"CP-{pump_id:03d}",
        initial_health=initial_health,
        design_flow=service['design_flow'] * random.uniform(0.9, 1.1),
        design_head=service['design_head'] * random.uniform(0.9, 1.1),
        design_speed=3000 + random.randint(-500, 500),
        fluid_density=service['density'],
        npsh_available=random.uniform(6, 12),
        env_model=env_model,
        enable_environmental=True,  
        enable_maintenance=False,  # Disable maintenance to increase failures
        enable_thermal_transients=True,
        enable_enhanced_vibration=True
    )
    
    logger.info(f"Simulating CP-{pump_id:03d} ({service['name']}, health: {initial_health})")
    
    # Simulate using extended function with degradation multiplier
    for record in simulate_equipment(
        pump, 
        pump_id, 
        'pump',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True
    ):
        if record['type'] == 'failure':
            pump_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
            logger.info(f"CP-{pump_id:03d} failed at {record['failure_time']} with code: {record['failure_mode_code']}")
            break
        else:
            pump_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                **record['state']
            })
    
    if i % 5 == 4:  # Print every 5 pumps
        print(f"  CP-{pump_id-4:03d} to CP-{pump_id:03d} completed")

print(f"\nTotal pump telemetry: {len(pump_telemetry):,} records")
print(f"Total pump failures: {len(pump_failures)}")

In [ ]:
# Combine all telemetry and failures
all_telemetry = turbine_telemetry + compressor_telemetry + pump_telemetry
all_failures = turbine_failures + compressor_failures + pump_failures


print("SIMULATION SUMMARY")
print(f"\nTotal telemetry records: {len(all_telemetry):,}")
print(f"  - Turbines:    {len(turbine_telemetry):,}")
print(f"  - Compressors: {len(compressor_telemetry):,}")
print(f"  - Pumps:       {len(pump_telemetry):,}")

print(f"\nTotal failures: {len(all_failures)}")
print(f"  - Turbines:    {len(turbine_failures)}")
print(f"  - Compressors: {len(compressor_failures)}")
print(f"  - Pumps:       {len(pump_failures)}")

# Failure modes analysis
if all_failures:
    print(f"\nFailure Modes:")
    failure_modes = {}
    for f in all_failures:
        mode = f['failure_mode_code']
        failure_modes[mode] = failure_modes.get(mode, 0) + 1
    
    for mode, count in sorted(failure_modes.items(), key=lambda x: x[1], reverse=True):
        print(f"  {mode}: {count}")


In [ ]:
# Convert to pandas DataFrames
df_telemetry = pd.DataFrame(all_telemetry)
df_failures = pd.DataFrame(all_failures)

print("\nTelemetry DataFrame:")
print(f"  Shape: {df_telemetry.shape}")
print(f"  Columns: {list(df_telemetry.columns[:10])}...")
print(f"  Memory usage: {df_telemetry.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

if not df_failures.empty:
    print("\nFailures DataFrame:")
    print(f"  Shape: {df_failures.shape}")
    print(f"  Columns: {list(df_failures.columns)}")

# Display sample telemetry
print("\nSample Telemetry (first 5 records):")
display(df_telemetry.head())

if not df_failures.empty:
    print("\nSample Failures:")
    display(df_failures.head())

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# Create database connection
try:
    db_url = os.getenv('POSTGRES_URL')
    engine = create_engine(db_url, echo=False)
    Session = sessionmaker(bind=engine)
    
    # Test connection
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
    
    print("Database connection successful")
    
except Exception as e:
    print(f"Database connection failed: {e}")
    print("\nPlease update DB_CONFIG with correct credentials and ensure PostgreSQL is running")
    raise

In [ ]:
# Separate telemetry by equipment type
df_turbine_telemetry = df_telemetry[df_telemetry['equipment_type'] == 'turbine'].copy()
df_compressor_telemetry = df_telemetry[df_telemetry['equipment_type'] == 'compressor'].copy()
df_pump_telemetry = df_telemetry[df_telemetry['equipment_type'] == 'pump'].copy()

print("Telemetry records by type:")
print(f"  Turbines:    {len(df_turbine_telemetry):,}")
print(f"  Compressors: {len(df_compressor_telemetry):,}")
print(f"  Pumps:       {len(df_pump_telemetry):,}")

In [ ]:
# Write turbine telemetry to database
if not df_turbine_telemetry.empty:
    # Rename equipment_id to turbine_id for database schema
    df_turbine_telemetry_db = df_turbine_telemetry.copy()
    df_turbine_telemetry_db.rename(columns={'equipment_id': 'turbine_id'}, inplace=True)
    df_turbine_telemetry_db.drop(columns=['equipment_type'], inplace=True)
    
    print(f"\nWriting {len(df_turbine_telemetry_db):,} turbine telemetry records to database...")
    
    try:
        df_turbine_telemetry_db.to_sql(
            name='gas_turbine_telemetry',
            schema='telemetry',
            con=engine,
            if_exists='append',
            index=False,
            chunksize=1000,
            method='multi'
        )
        print("Turbine telemetry written successfully")
    except Exception as e:
        print(f"Error writing turbine telemetry: {e}")

In [ ]:
# Write compressor telemetry to database
if not df_compressor_telemetry.empty:
    # Rename equipment_id to compressor_id for database schema
    df_compressor_telemetry_db = df_compressor_telemetry.copy()
    df_compressor_telemetry_db.rename(columns={'equipment_id': 'compressor_id'}, inplace=True)
    df_compressor_telemetry_db.drop(columns=['equipment_type'], inplace=True)
    
    print(f"\nWriting {len(df_compressor_telemetry_db):,} compressor telemetry records to database...")
    
    try:
        df_compressor_telemetry_db.to_sql(
            name='centrifugal_compressor_telemetry',
            schema='telemetry',
            con=engine,
            if_exists='append',
            index=False,
            chunksize=1000,
            method='multi'
        )
        print("Compressor telemetry written successfully")
    except Exception as e:
        print(f"Error writing compressor telemetry: {e}")

In [ ]:
# Write pump telemetry to database
if not df_pump_telemetry.empty:
    # Rename equipment_id to pump_id for database schema
    df_pump_telemetry_db = df_pump_telemetry.copy()
    df_pump_telemetry_db.rename(columns={'equipment_id': 'pump_id'}, inplace=True)
    df_pump_telemetry_db.drop(columns=['equipment_type'], inplace=True)
    
    print(f"\nWriting {len(df_pump_telemetry_db):,} pump telemetry records to database...")
    
    try:
        df_pump_telemetry_db.to_sql(
            name='centrifugal_pump_telemetry',
            schema='telemetry',
            con=engine,
            if_exists='append',
            index=False,
            chunksize=1000,
            method='multi'
        )
        print("Pump telemetry written successfully")
    except Exception as e:
        print(f"Error writing pump telemetry: {e}")

In [ ]:
# Write failure records to database
if not df_failures.empty:
    print(f"\nWriting {len(df_failures)} failure records to database...")
    
    # Prepare failure data for database
    df_failures_db = df_failures.copy()
    
    # Map equipment_type to specific ID column names
    type_mapping = {
        'turbine': 'turbine_id',
        'compressor': 'compressor_id',
        'pump': 'pump_id'
    }
    
    for eq_type, id_col in type_mapping.items():
        mask = df_failures_db['equipment_type'] == eq_type
        df_failures_db.loc[mask, id_col] = df_failures_db.loc[mask, 'equipment_id']
    
    # Convert state dict to JSON string if needed
    if 'state' in df_failures_db.columns:
        import json
        df_failures_db['state'] = df_failures_db['state'].apply(json.dumps)
    
    df_failures_db.drop(columns=['equipment_id'], inplace=True)
    
    try:
        df_failures_db.to_sql(
            name='equipment_failures',
            schema='telemetry',
            con=engine,
            if_exists='append',
            index=False,
            chunksize=100,
            method='multi'
        )
        print("Failure records written successfully")
    except Exception as e:
        print(f"Error writing failure records: {e}")
else:
    print("\nNo failures to write to database")

In [ ]:
from sqlalchemy import text

# Verify record counts in database
print("\nVerifying database writes...\n")

with engine.connect() as conn:
    # Count turbine telemetry
    result = conn.execute(text("SELECT COUNT(*) FROM telemetry.gas_turbine_telemetry"))
    turbine_count = result.scalar()
    print(f"  Turbine telemetry records in DB: {turbine_count:,}")
    
    # Count compressor telemetry
    result = conn.execute(text("SELECT COUNT(*) FROM telemetry.centrifugal_compressor_telemetry"))
    compressor_count = result.scalar()
    print(f"  Compressor telemetry records in DB: {compressor_count:,}")
    
    # Count pump telemetry
    result = conn.execute(text("SELECT COUNT(*) FROM telemetry.centrifugal_pump_telemetry"))
    pump_count = result.scalar()
    print(f"  Pump telemetry records in DB: {pump_count:,}")
    
    # Count failures
    try:
        result = conn.execute(text("SELECT COUNT(*) FROM telemetry.equipment_failures"))
        failure_count = result.scalar()
        print(f"  Failure records in DB: {failure_count}")
    except:
        print(f"  Failure records in DB: (table may not exist)")

print("\nDatabase verification complete")

In [ ]:
# Close database connection
engine.dispose()
print("Database connection closed")
print("SIMULATION COMPLETE")